[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/agentes-avanzados/02-mcp.ipynb)

# 02 — Model Context Protocol (MCP)

En este notebook exploramos el **Model Context Protocol (MCP)**: un estándar abierto que permite a los modelos de lenguaje interactuar con herramientas y recursos externos de forma estructurada.

Veremos dos enfoques complementarios:
- Un servidor MCP mínimo implementado en memoria.
- Tool use simulado directamente con la API de Anthropic (sin MCP), que ilustra el mismo concepto de forma más sencilla.

**Complemento del tutorial:** `tutoriales/agentes-avanzados/02-model-context-protocol.md`

**Requisitos:** `ANTHROPIC_API_KEY` en las variables de entorno.

In [ ]:
# %pip install mcp anthropic

import anthropic
import asyncio
import json
import os

cliente = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
MODELO = "claude-haiku-4-5-20251001"

print("Imports cargados correctamente.")

## 1. Qué es MCP

El **Model Context Protocol** es un protocolo cliente-servidor que estandariza cómo los modelos de IA acceden a herramientas, datos y recursos externos.

### Arquitectura

```
┌─────────────────────────────────────────────────────────┐
│                    APLICACIÓN HOST                       │
│                                                         │
│   ┌──────────────┐        ┌──────────────────────────┐  │
│   │    MODELO    │◄──────►│     CLIENTE MCP          │  │
│   │  (Claude)    │        │  (gestiona el protocolo) │  │
│   └──────────────┘        └──────────┬───────────────┘  │
└──────────────────────────────────────┼──────────────────┘
                                       │  Protocolo MCP
                          ┌────────────▼───────────────┐
                          │       SERVIDOR MCP          │
                          │                             │
                          │  • Herramientas (Tools)     │
                          │  • Recursos (Resources)     │
                          │  • Prompts (Prompts)        │
                          └─────────────────────────────┘
```

**Componentes principales:**

- **Cliente MCP**: Se ejecuta dentro de la aplicación host. Gestiona la conexión con uno o más servidores MCP.
- **Servidor MCP**: Proceso independiente que expone capacidades (herramientas, recursos, prompts) que el modelo puede usar.
- **Herramientas (Tools)**: Funciones que el modelo puede invocar (ej. buscar en la web, ejecutar código, consultar una base de datos).
- **Recursos (Resources)**: Datos que el servidor pone a disposición del modelo (ej. ficheros, registros de base de datos).
- **Prompts**: Plantillas de prompts reutilizables que el servidor ofrece.

### Por qué MCP en lugar de tool use nativo

| Característica | Tool use nativo | MCP |
|---|---|---|
| Estándar abierto | No | Sí |
| Reutilizable entre apps | No | Sí |
| Descubrimiento dinámico | No | Sí |
| Soporte multi-modelo | Depende | Sí |
| Complejidad inicial | Baja | Media |

## 2. Servidor MCP mínimo

Implementamos un servidor MCP en memoria con una herramienta `calcular` que suma dos números. Este servidor no usa stdio sino que se ejecuta directamente con asyncio para facilitar la experimentación en el notebook.

In [ ]:
# Servidor MCP mínimo en memoria con asyncio
# Nota: en producción los servidores MCP se ejecutan como procesos separados
# comunicados via stdio o HTTP. Aquí usamos un enfoque simplificado en memoria.

try:
    import mcp.server
    import mcp.types as types
    from mcp.server import Server
    MCP_DISPONIBLE = True
except ImportError:
    MCP_DISPONIBLE = False
    print("mcp no instalado. Ejecuta: pip install mcp")


async def demo_servidor_mcp():
    """Demuestra un servidor MCP mínimo con una herramienta de cálculo."""
    if not MCP_DISPONIBLE:
        print("Saltando demo de servidor MCP (librería no disponible).")
        return

    # Crear el servidor MCP
    servidor = Server("servidor-calculo")

    # Registrar el handler para listar herramientas
    @servidor.list_tools()
    async def listar_herramientas() -> list[types.Tool]:
        return [
            types.Tool(
                name="calcular",
                description="Suma dos números enteros y devuelve el resultado.",
                inputSchema={
                    "type": "object",
                    "properties": {
                        "a": {"type": "number", "description": "Primer número"},
                        "b": {"type": "number", "description": "Segundo número"},
                    },
                    "required": ["a", "b"],
                },
            )
        ]

    # Registrar el handler para ejecutar herramientas
    @servidor.call_tool()
    async def ejecutar_herramienta(
        nombre: str, argumentos: dict
    ) -> list[types.TextContent]:
        if nombre == "calcular":
            a = argumentos.get("a", 0)
            b = argumentos.get("b", 0)
            resultado = a + b
            return [
                types.TextContent(
                    type="text",
                    text=f"El resultado de {a} + {b} = {resultado}",
                )
            ]
        raise ValueError(f"Herramienta desconocida: {nombre}")

    # Simular invocación directa (en memoria, sin transporte stdio)
    print("Servidor MCP 'servidor-calculo' inicializado.")

    # Listar herramientas disponibles
    herramientas = await listar_herramientas()
    print(f"Herramientas disponibles: {[h.name for h in herramientas]}")

    # Invocar la herramienta calcular
    resultado = await ejecutar_herramienta("calcular", {"a": 42, "b": 58})
    print(f"Resultado de calcular(42, 58): {resultado[0].text}")


# Ejecutar la demo
asyncio.run(demo_servidor_mcp())

## 3. Tool use sin MCP (simulado)

Esta celda muestra el flujo completo de tool use con la API de Anthropic directamente, sin MCP. Es conceptualmente idéntico a MCP pero sin el protocolo de transporte. Ideal para entender la mecánica antes de añadir la capa MCP.

In [ ]:
# Simulación de tool use con la API de Anthropic (sin MCP)
# Flujo:
#   1. El usuario hace una pregunta que requiere buscar documentos.
#   2. Claude decide llamar a la herramienta buscar_documentos.
#   3. Nosotros ejecutamos la búsqueda localmente.
#   4. Devolvemos el resultado a Claude.
#   5. Claude formula la respuesta final.

# --- Definición de la herramienta ---
HERRAMIENTA_BUSCAR = {
    "name": "buscar_documentos",
    "description": (
        "Busca documentos en la base de conocimiento interna. "
        "Devuelve los fragmentos más relevantes para la consulta."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "consulta": {
                "type": "string",
                "description": "La consulta de búsqueda en lenguaje natural.",
            },
            "max_resultados": {
                "type": "integer",
                "description": "Número máximo de documentos a devolver (1-5).",
                "default": 3,
            },
        },
        "required": ["consulta"],
    },
}


# --- Implementación local de la herramienta ---
BASE_CONOCIMIENTO = [
    {"id": "doc001", "titulo": "Introducción a los LLMs", "contenido": "Los modelos de lenguaje grande (LLMs) son redes neuronales entrenadas en grandes cantidades de texto."},
    {"id": "doc002", "titulo": "Prompt engineering", "contenido": "El prompt engineering es el arte de diseñar instrucciones efectivas para guiar el comportamiento de los modelos de IA."},
    {"id": "doc003", "titulo": "RAG: Retrieval Augmented Generation", "contenido": "RAG combina recuperación de documentos con generación de texto para proporcionar respuestas basadas en fuentes específicas."},
    {"id": "doc004", "titulo": "Agentes de IA", "contenido": "Los agentes de IA son sistemas que pueden planificar, usar herramientas y completar tareas de múltiples pasos de forma autónoma."},
]


def buscar_documentos(consulta: str, max_resultados: int = 3) -> list[dict]:
    """Búsqueda simple por palabras clave en la base de conocimiento local."""
    palabras = consulta.lower().split()
    resultados = []
    for doc in BASE_CONOCIMIENTO:
        texto = (doc["titulo"] + " " + doc["contenido"]).lower()
        score = sum(1 for p in palabras if p in texto)
        if score > 0:
            resultados.append({**doc, "relevancia": score})
    resultados.sort(key=lambda x: x["relevancia"], reverse=True)
    return resultados[:max_resultados]


# --- Flujo completo de tool use ---
def chat_con_herramientas(pregunta: str) -> str:
    """
    Ejecuta una conversación con Claude que puede usar la herramienta buscar_documentos.
    Implementa el ciclo completo: petición → tool_use → resultado → respuesta final.
    """
    print(f"Pregunta: {pregunta}")
    print("-" * 50)

    mensajes = [{"role": "user", "content": pregunta}]

    # Primera llamada: Claude puede decidir usar una herramienta
    respuesta = cliente.messages.create(
        model=MODELO,
        max_tokens=500,
        tools=[HERRAMIENTA_BUSCAR],
        messages=mensajes,
    )

    # Verificar si Claude quiere usar una herramienta
    if respuesta.stop_reason == "tool_use":
        # Extraer la llamada a la herramienta
        bloque_tool = next(b for b in respuesta.content if b.type == "tool_use")
        herramienta_nombre = bloque_tool.name
        argumentos = bloque_tool.input

        print(f"Claude llama a: {herramienta_nombre}({argumentos})")

        # Ejecutar la herramienta localmente
        if herramienta_nombre == "buscar_documentos":
            docs_encontrados = buscar_documentos(
                argumentos["consulta"],
                argumentos.get("max_resultados", 3),
            )
            resultado_herramienta = json.dumps(docs_encontrados, ensure_ascii=False)
        else:
            resultado_herramienta = json.dumps({"error": "Herramienta no encontrada"})

        print(f"Resultado de la herramienta: {resultado_herramienta[:100]}...")

        # Segunda llamada: devolver el resultado a Claude
        mensajes = [
            {"role": "user", "content": pregunta},
            {"role": "assistant", "content": respuesta.content},
            {
                "role": "user",
                "content": [
                    {
                        "type": "tool_result",
                        "tool_use_id": bloque_tool.id,
                        "content": resultado_herramienta,
                    }
                ],
            },
        ]

        respuesta_final = cliente.messages.create(
            model=MODELO,
            max_tokens=500,
            tools=[HERRAMIENTA_BUSCAR],
            messages=mensajes,
        )
        return respuesta_final.content[0].text

    # Claude respondió directamente sin usar herramientas
    return respuesta.content[0].text


# Ejecutar el flujo completo
respuesta = chat_con_herramientas(
    "¿Qué documentos tenéis sobre agentes de IA y RAG?"
)
print("\nRespuesta final de Claude:")
print(respuesta)

## 4. Próximos pasos

Ahora que entiendes los fundamentos de MCP y tool use, puedes profundizar en:

1. **Tutorial completo de MCP**: Consulta `tutoriales/agentes-avanzados/02-model-context-protocol.md` para una explicación detallada de todos los conceptos.

2. **Servidor MCP con transporte stdio**: En producción, los servidores MCP se ejecutan como procesos independientes. Prueba a convertir el servidor de esta celda en un script que se comunique por stdin/stdout.

3. **Servidores MCP disponibles**: El ecosistema MCP crece rápidamente. Puedes encontrar servidores para GitHub, Slack, bases de datos, sistemas de ficheros y más en el repositorio oficial de Anthropic.

4. **Construye tu propio servidor**: Crea un servidor MCP que exponga una herramienta para consultar una API externa (clima, noticias, etc.) y úsala desde Claude.

```python
# Experimenta: añade una herramienta nueva al flujo de tool use
# Por ejemplo, una herramienta que devuelva la fecha y hora actuales:

from datetime import datetime

def obtener_fecha_hora() -> str:
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Intégrala en el flujo de chat_con_herramientas y prueba a preguntar:
# "¿Qué hora es ahora mismo?"
```